# 12 — Custom Paradigm Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/12_custom_paradigm.ipynb)

DantinoX’s **Paradigm** system is the extension point for new generation algorithms.

**What this notebook builds:** a **Semi-Autoregressive (SemiAR)** paradigm —
generates `block_size` tokens in parallel per step, left-to-right:

```
Step 0:  [▒ ▒ ▒ ▒ · · ·]        predict first block
Step 1:  [w₁ w₂ w₃ w₄ ▒ ▒ ▒ ·]   next block
```

**Sections:** interface · implementation · training · generation · block-size ablation · registry · extension ideas

**Runtime**: ~20 min · GPU (T4)

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import jax

print('Devices:', jax.devices())

In [ ]:
!pip install -q "dantinox[all] @ git+https://github.com/winstonsmith1897/DantinoX.git"

In [ ]:
import os
import urllib.request

if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

## 1 — The Paradigm interface

```python
class BaseParadigm(abc.ABC):
    def build_model(self, rngs: nnx.Rngs): ...      # required
    def loss(self, model, batch, rng): ...           # required
    def generate(self, model, *, max_new_tokens, **kw): ...  # required
    def stream(self, model, *, max_new_tokens, **kw): ...    # required
```

`Trainer` only calls `build_model()` and `loss()`.
`stream()` / `generate()` are inference-only.

## 2 — SemiAR design

**Training**: randomly mask `mask_rate` tokens, train to predict them (same as discrete diffusion).

**Generation**:
```
tokens = [MASK] * T
for block in range(T // block_size):
    s, e = block*block_size, (block+1)*block_size
    logits       = model(tokens)        # full bidirectional pass
    tokens[s:e]  = argmax(logits[s:e])  # decode this block only
```

Larger `block_size` → fewer steps → faster but less coherent.

In [ ]:
from collections.abc import Iterator

import jax
import jax.numpy as jnp

import dantinox as dx

BASE_CFG = dx.ModelConfig(
    paradigm='discrete', dim=128, n_heads=4, num_blocks=4,
    causal=False, dropout=0.1,
)


class SemiARParadigm(dx.Paradigm):
    """Semi-autoregressive: decodes one block per step, left to right.

    Parameters
    ----------
    model_config : dx.ModelConfig   Must have causal=False.
    block_size   : int              Tokens decoded per step.
    mask_rate    : float            Fraction masked during training.
    """

    def __init__(self, model_config, block_size=4, mask_rate=0.5):
        super().__init__(model_config)
        self.block_size    = block_size
        self.mask_rate     = mask_rate
        self.mask_token_id = 1   # updated after tokenizer is loaded

    def build_model(self, rngs):
        from dantinox.core.model import Transformer
        return Transformer(self.model_config, rngs=rngs)

    def loss(self, model, batch, rng):
        B, T   = batch.shape
        rng, s = jax.random.split(rng)
        mask   = jax.random.uniform(s, (B, T)) < self.mask_rate
        x_t    = jnp.where(mask, self.mask_token_id, batch)
        logits = model(x_t, deterministic=False).logits
        lp     = jax.nn.log_softmax(logits, -1)
        nll    = -lp[jnp.arange(B)[:,None], jnp.arange(T)[None,:], batch]
        return (nll * mask).sum() / jnp.maximum(mask.sum(), 1.0)

    def stream(self, model, *, rng=None, max_new_tokens=64, **kwargs) -> Iterator:
        if rng is None: rng = jax.random.PRNGKey(0)
        n_blocks = max_new_tokens // self.block_size
        tokens   = jnp.full((1, max_new_tokens), self.mask_token_id, jnp.int32)
        for i in range(n_blocks):
            s, e    = i * self.block_size, (i+1) * self.block_size
            logits  = model(tokens, deterministic=True).logits
            new_tok = jnp.argmax(logits[0, s:e, :], axis=-1)
            tokens  = tokens.at[0, s:e].set(new_tok)
            yield i, n_blocks, tokens

    def generate(self, model, *, rng=None, max_new_tokens=64, **kwargs):
        result = None
        for _, _, tok in self.stream(model, rng=rng, max_new_tokens=max_new_tokens):
            result = tok
        return result


print('SemiARParadigm defined.')

## 3 — Training

`dx.Trainer` only needs `build_model()` and `loss()`.

In [ ]:
semiar    = SemiARParadigm(BASE_CFG, block_size=4, mask_rate=0.5)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=2, batch_size=16, tokenizer_type='char')
run_dir   = dx.Trainer(semiar, train_cfg).fit('tiny_shakespeare.txt')
print('Checkpoint:', run_dir)

## 4 — Streaming generation

Each printed line shows the revealed tokens after one more block is decoded.

In [ ]:
import os

from dantinox.utils.tokenizer import load_tokenizer_from_file

model = dx.load(run_dir, paradigm=semiar)
tok   = load_tokenizer_from_file(os.path.join(run_dir, 'tokenizer.json'))
if hasattr(tok, 'mask_token_id'):
    semiar.mask_token_id = tok.mask_token_id

MAX_NEW = 32
print(f'Generating {MAX_NEW} tokens  (block_size={semiar.block_size})\n')
for step, total, tokens in semiar.stream(model, max_new_tokens=MAX_NEW):
    revealed = tok.decode(tokens[0, :(step+1)*semiar.block_size].tolist())
    print(f'Block {step+1:2d}/{total} │ {revealed!r}')
print('\nFinal:', tok.decode(tokens[0].tolist()))

## 5 — Block-size ablation

| `block_size` | Steps | Behaviour |
|---|---|---|
| `1` | T | One token at a time — iterative masked LM |
| `4–8` | T/4–T/8 | Sweet spot: fast + quality |
| `T` | 1 | One-shot — single-step discrete diffusion |

In [ ]:
import time

import pandas as pd

MAX_TOKENS, rows = 64, []
for bsz in (1, 4, 8, 16, MAX_TOKENS):
    p = SemiARParadigm(BASE_CFG, block_size=bsz)
    m = dx.load(run_dir, paradigm=p)
    p.mask_token_id = semiar.mask_token_id
    t0 = time.perf_counter()
    for _ in range(5): p.generate(m, max_new_tokens=MAX_TOKENS)
    elapsed = (time.perf_counter()-t0)/5
    rows.append({'block_size':bsz,'n_steps':MAX_TOKENS//bsz,
                 'time_s':round(elapsed,4),'tok_s':round(MAX_TOKENS/elapsed,1)})
print(pd.DataFrame(rows).set_index('block_size').to_string())

## 6 — Registering in the paradigm registry

Enables `dx.fit('semiar', ...)` and `dx.Paradigm(ModelConfig(paradigm='semiar'))`.

In [ ]:
try:
    from dantinox.core.paradigm_registry import register_paradigm

    @register_paradigm('semiar')
    class SemiARRegistered(SemiARParadigm): pass

    print("Registered 'semiar'.")
    if hasattr(dx, 'list_paradigms'):
        print('Available paradigms:', dx.list_paradigms())

    # run = dx.fit('semiar', 'tiny_shakespeare.txt', dim=128, n_heads=4, num_blocks=4)
    # p   = dx.Paradigm(dx.ModelConfig(paradigm='semiar'))
    # m   = dx.load(run, paradigm=p)
except ImportError:
    print('registry API not available in this version — skipped.')

## 7 — Extension ideas

**Temperature sampling** instead of argmax:
```python
logits_b = logits[0, s:e, :] / temperature
new_tok  = jax.random.categorical(rng, jnp.log(jax.nn.softmax(logits_b, -1)))
```

**Confidence gating** — re-mask low-confidence tokens:
```python
conf    = jax.nn.softmax(logits[0, s:e], -1).max(-1)
new_tok = jnp.where(conf > threshold, argmax_tok, mask_id)
```

**Block-structured training** — always mask future blocks:
```python
start = jax.random.randint(rng, (), 0, T // block_size) * block_size
mask  = jnp.arange(T) >= start
```

**LoRA adapters** — works out of the box:
```python
lora_cfg = dataclasses.replace(BASE_CFG, use_lora=True, lora_rank=8, lora_targets='all')
semiar   = SemiARParadigm(lora_cfg, block_size=4)
```

**Swap backbone** for GQA or MoE:
```python
class SemiAR_GQA(SemiARParadigm):
    def build_model(self, rngs):
        from dantinox.core.model import Transformer
        cfg = dataclasses.replace(self.model_config, attention='gqa', kv_heads=2)
        return Transformer(cfg, rngs=rngs)
```